In [79]:
from gensim.models import Word2Vec
import pandas as pd
import pickle


with open('/home/peng/PycharmProjects/featurematch/data/processed/final/ids/omim_id_name.dict', 'rb') as handle:
    omim_id_name = pickle.load(handle)
    
# with open('/home/peng/PycharmProjects/featurematch/data/processed/final/ids/gene_id_name.dict', 'rb') as handle:
#     gene_id_name = pickle.load(handle)

with open('/home/peng/PycharmProjects/featurematch/data/processed/final/ids/hpo_id_name.dict', 'rb') as handle:
    hpo_id_name = pickle.load(handle)
    
model = Word2Vec.load('../../../model/withoutgene_omim_without_patients/without_patients.model')


In [80]:
evaluation_save = list()
evaluation_vis = list()

evaluation_patients = '/home/peng/PycharmProjects/featurematch/data/processed/patients/reform/evaluation/evaluation_pedia.tsv'
with open(evaluation_patients, 'r') as e_file:
    content = e_file.read().splitlines()
    content = [x.split('\t') for x in content]
    for line in content:
        current_patient=[]
        feature_filtered_nodes= []     
        patient_id = line[0]
        disease_id = line[1]
        features = line[2].split(',')
        similar_nodes = model.most_similar(positive=features, topn=20000)
        
        for node in similar_nodes:
            if node[0].startswith('OMIM'):
                feature_filtered_nodes.append(node[0])
        rank = feature_filtered_nodes.index(disease_id) + 1
        evaluation_save.append([patient_id, disease_id, ','.join(features), str(len(features)),','.join(feature_filtered_nodes[:10]), str(rank)])    
        evaluation_vis.append([patient_id, omim_id_name[disease_id], ','.join([hpo_id_name[feature] for feature in features]), str(len(features)), [omim_id_name[omim_id] for omim_id in feature_filtered_nodes[:10]], str(rank)])

/home/peng/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:14: DeprecationWarning: Call to deprecated `most_similar` (Method will be removed in 4.0.0, use self.wv.most_similar() instead).
  


In [81]:
pd.set_option('display.max_colwidth', -1)
saveframe = pd.DataFrame(evaluation_save, columns = ['patient_id', 'omim_id', 'features', 'No.features', 'results_hpo', 'rank_hpo'])
saveframe.to_csv('../../../model/withoutgene_omim_without_patients/evaluation.tsv', sep='\t', index=None)

In [82]:
visframe = pd.DataFrame(evaluation_vis, columns = ['patient_id', 'omim_id', 'features', 'No.features', 'results_hpo', 'rank_hpo'])
visframe.to_excel('../../../model/withoutgene_omim_without_patients/evaluation.xlsx', index=None)